## Данные 
Данные представлены с трех независимых источников и состоят из следующих признаков:
- Unnamed: 0 - ID строки
- topic_id - идентификационный номер топика. Топик состоит из одного вопроса и всех ответов  к нему
- id — идентификационный номер поста
- parent_id — идентификационный номер родительского поста для ответов или -1 для вопросов
- post_type — тип поста. Посты бывают только двух типов: вопрос — 1, ответ — 2.
- created_at — дата создания публикации
- author_id — идентификационный номер автора поста
- post_count - количество постов в вопросе
- view_count - количество просмотров
- reply_time - время ответа
- is_accepted_answer - ответ на вопрос?
- author_username - имя пользователя

In [3]:
import pandas as pd
import datetime as dt
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

sns.set(style="darkgrid")

Загружаем скопом все данные

In [4]:
full_datas = {}
folder_path = r'..\data\raw_data'
csv_files = glob.glob(os.path.join(folder_path, '*.csv'))
for file_path in csv_files:
    # Получаем имя файла без расширения
    filename = os.path.splitext(os.path.basename(file_path))[0]
    
    # Читаем CSV
    full_datas[filename] = pd.read_csv(file_path)

for name, data in full_datas.items():
    data.drop(data[data['author_id'] < 0].index, inplace=True)
    data['created_at'] = pd.to_datetime(data['created_at'], format='mixed', errors='coerce')

In [5]:
full_datas['acer'].head(10)

,topic_id,id,parent_id,post_type,posts_count,view_count,created_at,reply_time,is_accepted_answer,author_id,author_username
0,28333,1035536,-1,1,5,1,2006-10-05 22:51:18+00:00,0.00,False,11738,OnegirlOneboy
1,28333,1035560,1035536,2,0,0,2006-10-05 23:50:00+00:00,0.04,False,4163,toorobbin
3,28333,1035565,1035536,2,0,0,2006-10-06 00:02:29+00:00,0.05,False,4163,toorobbin
4,28333,1035572,1035536,2,0,0,2006-10-06 01:14:36+00:00,0.10,False,11738,OnegirlOneboy
6,35084,1297449,-1,1,7,1,2006-10-02 21:26:51+00:00,0.00,False,6,Amy
7,35084,1297450,1297449,2,0,0,2006-10-03 00:44:07+00:00,0.14,False,10481,babygurl2
8,35084,1297460,1297449,2,0,0,2006-10-03 01:42:16+00:00,0.18,False,6,Amy
9,35084,1297529,1297449,2,0,0,2006-10-04 13:30:45+00:00,1.67,False,10068,katrina_woodroffe
10,35084,1297540,1297449,2,0,0,2006-10-04 14:50:16+00:00,1.72,False,6,Amy
11,35084,1297584,1297449,2,0,0,2006-10-05 15:38:00+00:00,2.76,False,10068,katrina_woodroffe


Метрики, которые нужно рассчитать
MAU - Количество уникальных пользователей с хотя бы одним активным действием за месяц.

D30 Retention - Процент пользователей, вернувшихся через 30 дней после первой активности.

Total Number of Actions - Общее количество действий за месяц

Number of Posts/Discussions - Количество новых обсуждений или вопросов. - Количество новых обсуждений или вопросов.

Retention

User Churn - Процент пользователей, переставших быть активными. - Процент пользователей, переставших быть активными.

Rolling Retention

Stickiness

Network Density / Reciprocity

Percentage of New Users Receiving First Reply - Процент новичков, получивших хотя бы один ответ.

Median Time to First Reply - Медианное время до первого сообщения в вопросе

Median Time to Be Answered - Медианное время до пометки вопрос решен

Percentage of New Users Returning Within 7 Days - Процент новичков, вернувшихся в течение 7 дней.

Unanswered Rate - Процент вопросов без ответа.

Average Replies per Discussion - Среднее число ответов на обсуждение.

Number of Regular Users - Количество регулярно активных пользователей.

Core Contribution Share - Доля контента от top X% пользователей.

Reactivated Users Count - Количество пользователей, вернувшихся после периода неактивности.

In [4]:
def month_calendar(data):
    data = data.copy()
    data['created_at'] = pd.to_datetime(data['created_at'])

    months = pd.period_range(
        start=data['created_at'].min().to_period('M'),
        end=data['created_at'].max().to_period('M'),
        freq='M'
    )

    return pd.DataFrame({'month': months})

In [5]:
#Создадим функцию, которая считает одну метрику и возвращает Series с результатми по месяцам
def MAU(data):
    data = data.copy()
    data['month'] = data['created_at'].dt.to_period('M')
    #нужно добавить название столбца
    mau = data.groupby('month')['author_id'].nunique().reset_index(name='mau')
    return mau

mau = MAU(full_datas['elastic'])

In [6]:
display(mau)

,month,mau
0,2010-07,2
1,2010-10,2
2,2011-02,4
3,2011-03,1
4,2011-04,2
...,...,...
163,2024-10,686
164,2024-11,407
165,2024-12,468
166,2025-01,469


In [7]:
#Процент пользователей, вернувшихся через 30 дней после первой активности. Возвращает такой же Series, как и MAU
# Попробовать связать с теми, кому не ответили в течении первых 20 часов из прошлой гипотезы
def new_user_d7_retention(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['activity_date'] = data['created_at'].dt.normalize()

    first_activity = (
        data.groupby('author_id')['activity_date']
        .min()
        .reset_index()
        .rename(columns={'activity_date': 'first_activity'})
    )

    first_activity['cohort_month'] = first_activity['first_activity'].dt.to_period('M')

    data = data.merge(first_activity, on='author_id')

    data['days_from_start'] = (
        data['activity_date'] - data['first_activity']
    ).dt.days

    cohort_size = (
        first_activity.groupby('cohort_month')['author_id']
        .nunique()
    )

    retained_d7 = (
        data[data['days_from_start'] == 7]
        .groupby('cohort_month')['author_id']
        .nunique()
    )

    result = (
        (retained_d7 / cohort_size)
        .fillna(0)
        .reset_index(name='d7')
        .rename(columns={'cohort_month': 'month'})
    )

    result['d7'] = result['d7'] * 100

    return result
    

d7 = new_user_d7_retention(full_datas['acer'])
    

In [8]:
#Процент пользователей, вернувшихся через 30 дней после первой активности. Возвращает такой же Series, как и MAU
# Попробовать связать с теми, кому не ответили в течении первых 20 часов из прошлой гипотезы
def new_user_d30_retention(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['activity_date'] = data['created_at'].dt.normalize()

    first_activity = (
        data.groupby('author_id')['activity_date']
        .min()
        .reset_index()
        .rename(columns={'activity_date': 'first_activity'})
    )

    first_activity['cohort_month'] = first_activity['first_activity'].dt.to_period('M')

    data = data.merge(first_activity, on='author_id')

    data['days_from_start'] = (
        data['activity_date'] - data['first_activity']
    ).dt.days

    cohort_size = (
        first_activity.groupby('cohort_month')['author_id']
        .nunique()
    )

    retained_d30 = (
        data[data['days_from_start'] == 30]
        .groupby('cohort_month')['author_id']
        .nunique()
    )

    result = (
        (retained_d30 / cohort_size)
        .fillna(0)
        .reset_index(name='d30')
        .rename(columns={'cohort_month': 'month'})
    )

    result['d30'] = result['d30'] * 100

    return result
    

d30 = new_user_d30_retention(full_datas['acer'])
    

In [9]:
#Monthly Retention = пользователи, активные в месяце M и снова активные в месяце M+1

def monthly_retention(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    user_months = (
        data[['author_id', 'month']]
        .drop_duplicates()
        .sort_values(['author_id', 'month'])
    )

    current_month = user_months.copy()
    current_month['next_month'] = current_month['month'] + 1

    next_month_activity = user_months.rename(
        columns={'month': 'next_month'}
    )

    retained = current_month.merge(
        next_month_activity,
        on=['author_id', 'next_month'],
        how='inner'
    )

    active_users = (
        current_month.groupby('month')['author_id']
        .nunique()
    )

    retained_users = (
        retained.groupby('month')['author_id']
        .nunique()
    )

    result = (
        (retained_users / active_users)
        .fillna(0)
        .reset_index(name='monthly_retention')
    )

    result['monthly_retention'] = result['monthly_retention'] * 100

    return result

monthly_retention = monthly_retention(full_datas['acer'])


In [10]:
def total_number_of_actions(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    result = (
        data.groupby('month')
        .size()
        .reset_index(name='total_number_of_actions')
    )

    return result

total_number_of_actions = total_number_of_actions(full_datas['acer'])

In [11]:
def number_of_posts_discussions(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    result = (
        data[data['post_type'] == 1]
        .groupby('month')
        .size()
        .reset_index(name='number_of_posts_discussions')
    )

    return result

number_of_posts_discussions = number_of_posts_discussions(full_datas['acer'])

In [12]:
# количество пользователей, активных в этом месяце, не были активны в следующем месяце.
def user_churn(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    user_months = (
        data[['author_id', 'month']]
        .drop_duplicates()
        .sort_values(['author_id', 'month'])
    )

    current_month = user_months.copy()
    current_month['next_month'] = current_month['month'] + 1

    next_month_activity = user_months.rename(
        columns={'month': 'next_month'}
    )

    retained = current_month.merge(
        next_month_activity,
        on=['author_id', 'next_month'],
        how='inner'
    )

    active_users = (
        current_month.groupby('month')['author_id']
        .nunique()
    )

    retained_users = (
        retained.groupby('month')['author_id']
        .nunique()
    )

    retention = (retained_users / active_users).fillna(0)

    result = (
        (1 - retention)
        .reset_index(name='user_churn')
    )

    result['user_churn'] = result['user_churn'] * 100 

    return result

user_churn = user_churn(full_datas['acer'])

In [13]:
def percentage_new_users_receiving_first_reply(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    first_posts = (
        data.sort_values('created_at')
        .groupby('author_id')
        .first()
        .reset_index()
    )

    first_questions = first_posts[first_posts['post_type'] == 1].copy()

    first_questions = first_questions.rename(
        columns={
            'id': 'first_post_id',
            'month': 'first_month'
        }
    )

    replies = (
        data[data['post_type'] == 2][['parent_id']]
        .drop_duplicates()
        .rename(columns={'parent_id': 'replied_to_post_id'})
    )

    replied_first_questions = first_questions.merge(
        replies,
        left_on='first_post_id',
        right_on='replied_to_post_id',
        how='left'
    )

    replied_first_questions['received_first_reply'] = (
        replied_first_questions['replied_to_post_id'].notna()
    )

    result = (
        replied_first_questions
        .groupby('first_month')['received_first_reply']
        .mean()
        .mul(100)
        .reset_index(name='percentage_new_users_receiving_first_reply')
        .rename(columns={'first_month': 'month'})
    )

    return result


percentage_new_users_receiving_first_reply = percentage_new_users_receiving_first_reply(full_datas['acer'])

In [14]:
def median_time_to_first_reply(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    questions = (
        data[data['post_type'] == 1][['id', 'created_at', 'month']]
        .rename(columns={
            'id': 'question_id',
            'created_at': 'question_created_at',
            'month': 'question_month'
        })
    )

    replies = (
        data[data['post_type'] == 2][['parent_id', 'created_at']]
        .rename(columns={
            'parent_id': 'question_id',
            'created_at': 'reply_created_at'
        })
    )

    first_replies = (
        replies.groupby('question_id')['reply_created_at']
        .min()
        .reset_index()
    )

    question_replies = questions.merge(
        first_replies,
        on='question_id',
        how='inner'
    )

    question_replies['time_to_first_reply_hours'] = (
        question_replies['reply_created_at']
        - question_replies['question_created_at']
    ).dt.total_seconds() / 3600

    result = (
        question_replies
        .groupby('question_month')['time_to_first_reply_hours']
        .median()
        .reset_index(name='median_time_to_first_reply_hours')
        .rename(columns={'question_month': 'month'})
    )

    return result

median_time_to_first_reply = median_time_to_first_reply(full_datas['elastic'])


In [15]:
def median_time_to_be_answered(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    questions = (
        data[data['post_type'] == 1][['id', 'created_at', 'month']]
        .rename(columns={
            'id': 'question_id',
            'created_at': 'question_created_at',
            'month': 'question_month'
        })
    )

    accepted_answers = (
        data[
            (data['post_type'] == 2) &
            (data['is_accepted_answer'] == True)
        ][['parent_id', 'created_at']]
        .rename(columns={
            'parent_id': 'question_id',
            'created_at': 'answered_at'
        })
    )

    first_accepted_answers = (
        accepted_answers.groupby('question_id')['answered_at']
        .min()
        .reset_index()
    )

    question_answers = questions.merge(
        first_accepted_answers,
        on='question_id',
        how='inner'
    )

    question_answers['time_to_be_answered_hours'] = (
        question_answers['answered_at']
        - question_answers['question_created_at']
    ).dt.total_seconds() / 3600

    result = (
        question_answers
        .groupby('question_month')['time_to_be_answered_hours']
        .median()
        .reset_index(name='median_time_to_be_answered_hours')
        .rename(columns={'question_month': 'month'})
    )

    return result

median_time_to_be_answered = median_time_to_be_answered(full_datas['elastic'])


In [16]:
#unanswered_rate - доля пустых вопросов
def unanswered_rate(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    questions = (
        data[data['post_type'] == 1][['id', 'month']]
        .rename(columns={'id': 'question_id'})
    )

    replied_questions = (
        data[data['post_type'] == 2][['parent_id']]
        .drop_duplicates()
        .rename(columns={'parent_id': 'question_id'})
    )

    questions = questions.merge(
        replied_questions,
        on='question_id',
        how='left',
        indicator=True
    )

    questions['is_unanswered'] = questions['_merge'] == 'left_only'

    result = (
        questions.groupby('month')['is_unanswered']
        .mean()
        .mul(100)
        .reset_index(name='unanswered_rate')
    )

    return result

unanswered_rate = unanswered_rate(full_datas['elastic'])

In [17]:
def average_replies_per_topic(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    questions = (
        data[data['post_type'] == 1][['id', 'month']]
        .rename(columns={
            'id': 'question_id',
            'month': 'question_month'
        })
    )

    replies_count = (
        data[data['post_type'] == 2]
        .groupby('parent_id')
        .size()
        .reset_index(name='replies_count')
        .rename(columns={'parent_id': 'question_id'})
    )

    questions_replies = questions.merge(
        replies_count,
        on='question_id',
        how='left'
    )

    questions_replies['replies_count'] = questions_replies['replies_count'].fillna(0)

    result = (
        questions_replies
        .groupby('question_month')['replies_count']
        .mean()
        .reset_index(name='average_replies_per_discussion')
        .rename(columns={'question_month': 'month'})
    )

    return result

average_replies_per_topic= average_replies_per_topic(full_datas['elastic'])

In [18]:
#Regular User = пользователь, который был активен минимум N дней в месяце
#Number of Regular Users = количество пользователей с активностью >= 3 разных дней в месяце
def number_of_regular_users(data, min_active_days=3):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')
    data['date'] = data['created_at'].dt.date

    user_activity = (
        data
        .groupby(['month', 'author_id'])['date']
        .nunique()
        .reset_index(name='active_days')
    )

    regular_users = user_activity[
        user_activity['active_days'] >= min_active_days
    ]

    result = (
        regular_users
        .groupby('month')['author_id']
        .nunique()
        .reset_index(name='number_of_regular_users')
    )

    return result

number_of_regular_users = number_of_regular_users(full_datas['acer'])

In [19]:
#Core Contribution Share = контент от top X% самых активных пользователей / весь контент
def core_contribution_share(data, top_percent=0.05):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    monthly_user_content = (
        data
        .groupby(['month', 'author_id'])
        .size()
        .reset_index(name='content_count')
    )

    results = []

    for month, month_data in monthly_user_content.groupby('month'):
        month_data = month_data.sort_values(
            'content_count',
            ascending=False
        )

        top_n = max(1, int(len(month_data) * top_percent))

        top_content = month_data.head(top_n)['content_count'].sum()
        total_content = month_data['content_count'].sum()

        share = top_content / total_content * 100

        results.append({
            'month': month,
            'core_contribution_share': share
        })

    return pd.DataFrame(results)

core_contribution_share = core_contribution_share(full_datas['acer'])

In [20]:
#Reactivated User = пользователь, который был активен в текущем месяце, но не был активен в предыдущие 2 месяца, и при этом уже был активен раньше.
def reactivated_users_count(data, inactive_months=2):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    monthly_users = (
        data[['month', 'author_id']]
        .drop_duplicates()
        .sort_values(['author_id', 'month'])
    )

    results = []

    all_months = sorted(monthly_users['month'].unique())

    for month in all_months:
        current_users = set(
            monthly_users[monthly_users['month'] == month]['author_id']
        )

        inactive_period = [
            month - i
            for i in range(1, inactive_months + 1)
        ]

        inactive_period_users = set(
            monthly_users[
                monthly_users['month'].isin(inactive_period)
            ]['author_id']
        )

        previous_users = set(
            monthly_users[
                monthly_users['month'] < min(inactive_period)
            ]['author_id']
        )

        reactivated_users = (
            current_users
            - inactive_period_users
        ) & previous_users

        results.append({
            'month': month,
            'reactivated_users_count': len(reactivated_users)
        })

    return pd.DataFrame(results)

reactivated_users_count = reactivated_users_count(full_datas['acer'])

In [7]:
#data - DataFrame с колонкой month и метрикой, metric_column - имя метрики
def add_mom_yoy(data, metric_column=None):
    data = data.copy()

    if 'month' not in data.columns:
        raise ValueError(f"Column 'month' not found. Available columns: {list(data.columns)}")

    if metric_column is None:
        metric_columns = [col for col in data.columns if col != 'month']

        if len(metric_columns) != 1:
            raise ValueError(
                f"Could not detect metric column automatically. "
                f"Found metric columns: {metric_columns}. "
                f"Pass metric_column manually."
            )

        metric_column = metric_columns[0]

    data['month'] = data['month'].astype(str)
    data['month'] = pd.PeriodIndex(data['month'], freq='M')

    data = data.sort_values('month').reset_index(drop=True)

    for period, suffix in [(1, 'mom'), (12, 'yoy')]:
        previous = data[metric_column].shift(period)
        current = data[metric_column]

        change = ((current - previous) / previous) * 100

        change = change.mask((previous == 0) & (current > 0), 100)
        change = change.mask((previous == 0) & (current == 0), 0)

        change = change.replace([np.inf, -np.inf], np.nan)

        data[f'{metric_column}_{suffix}'] = change

    return data

In [22]:
#Resolution Rate = доля вопросов с accepted answer
def resolution_rate(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    questions = (
        data[data['post_type'] == 1][['id', 'month']]
        .rename(columns={
            'id': 'question_id',
            'month': 'question_month'
        })
    )

    accepted_answers = (
        data[
            (data['post_type'] == 2) &
            (data['is_accepted_answer'] == True)
        ][['parent_id']]
        .drop_duplicates()
        .rename(columns={'parent_id': 'question_id'})
    )

    questions_with_status = questions.merge(
        accepted_answers,
        on='question_id',
        how='left',
        indicator=True
    )

    questions_with_status['is_resolved'] = (
        questions_with_status['_merge'] == 'both'
    )

    result = (
        questions_with_status
        .groupby('question_month')['is_resolved']
        .mean()
        .mul(100)
        .reset_index(name='resolution_rate')
        .rename(columns={'question_month': 'month'})
    )

    return result

resolution_rate = resolution_rate(full_datas['elastic'])


In [23]:
#Answered but Unresolved Rate - Доля вопросов с ответами, но без принятого решения
def answered_but_unresolved_rate(data):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    questions = (
        data[data['post_type'] == 1][['id', 'month']]
        .rename(columns={
            'id': 'question_id',
            'month': 'question_month'
        })
    )

    replies = (
        data[data['post_type'] == 2][['parent_id']]
        .drop_duplicates()
        .rename(columns={'parent_id': 'question_id'})
    )

    data['is_accepted_answer_clean'] = (
        data['is_accepted_answer']
        .fillna(False)
        .astype(str)
        .str.lower()
        .isin(['true', '1', '1.0'])
    )

    accepted_answers = (
        data[
            (data['post_type'] == 2) &
            (data['is_accepted_answer_clean'])
        ][['parent_id']]
        .drop_duplicates()
        .rename(columns={'parent_id': 'question_id'})
    )

    questions_status = (
        questions
        .merge(replies.assign(has_reply=True), on='question_id', how='left')
        .merge(accepted_answers.assign(is_resolved=True), on='question_id', how='left')
    )

    questions_status['has_reply'] = questions_status['has_reply'].fillna(False)
    questions_status['is_resolved'] = questions_status['is_resolved'].fillna(False)

    questions_status['answered_but_unresolved'] = (
        questions_status['has_reply'] &
        ~questions_status['is_resolved']
    )

    result = (
        questions_status
        .groupby('question_month')['answered_but_unresolved']
        .mean()
        .mul(100)
        .reset_index(name='answered_but_unresolved_rate')
        .rename(columns={'question_month': 'month'})
    )

    return result

answered_but_unresolved_rate = answered_but_unresolved_rate(full_datas['acer'])
display(answered_but_unresolved_rate)

,month,answered_but_unresolved_rate
0,2006-08,92.258065
1,2006-09,88.130334
2,2006-10,88.289206
3,2006-11,89.645777
4,2006-12,92.323097
...,...,...
149,2025-05,89.477212
150,2025-06,90.783410
151,2025-07,88.955823
152,2025-08,85.609103


In [24]:
def new_user_activation_rate(data, days=7):
    data = data.copy()

    data['created_at'] = pd.to_datetime(data['created_at'])
    data['month'] = data['created_at'].dt.to_period('M')

    first_activity = (
        data
        .groupby('author_id')['created_at']
        .min()
        .reset_index(name='first_activity_at')
    )

    first_activity['first_month'] = (
        first_activity['first_activity_at']
        .dt.to_period('M')
    )

    user_actions = data.merge(
        first_activity,
        on='author_id',
        how='left'
    )

    user_actions_after_first = user_actions[
        (user_actions['created_at'] > user_actions['first_activity_at']) &
        (user_actions['created_at'] <= user_actions['first_activity_at'] + pd.Timedelta(days=days))
    ]

    activated_users = (
        user_actions_after_first[['author_id']]
        .drop_duplicates()
        .assign(is_activated=True)
    )

    new_users = first_activity.merge(
        activated_users,
        on='author_id',
        how='left'
    )

    new_users['is_activated'] = (
        new_users['is_activated']
        .fillna(False)
    )

    result = (
        new_users
        .groupby('first_month')['is_activated']
        .mean()
        .mul(100)
        .reset_index(name=f'new_user_activation_rate_{days}d')
        .rename(columns={'first_month': 'month'})
    )

    return result

new_user_activation_rate_7d = new_user_activation_rate(full_datas['acer'], days=7)
display(new_user_activation_rate_7d)


,month,new_user_activation_rate_7d
0,2006-08,51.894563
1,2006-09,51.485149
2,2006-10,51.403061
3,2006-11,51.470588
4,2006-12,53.846154
...,...,...
197,2025-05,40.740741
198,2025-06,28.000000
199,2025-07,29.487179
200,2025-08,40.579710


In [25]:
display(resolution_rate.sample(10))

,month,resolution_rate
15,2012-04,0.000000
85,2019-02,0.000000
43,2014-11,0.000000
78,2018-02,0.000000
150,2024-08,19.026549
60,2016-04,3.141361
128,2022-10,23.345259
117,2021-11,23.020408
9,2011-09,0.000000
92,2019-10,25.000000


In [26]:
mau_mom_yoy = add_mom_yoy(mau)
display(mau_mom_yoy)

,month,mau,mau_mom,mau_yoy
0,2010-07,2,NaN,NaN
1,2010-10,2,0.000000,NaN
2,2011-02,4,100.000000,NaN
3,2011-03,1,-75.000000,NaN
4,2011-04,2,100.000000,NaN
...,...,...,...,...
163,2024-10,686,8.201893,-18.138425
164,2024-11,407,-40.670554,-51.662708
165,2024-12,468,14.987715,-36.671177
166,2025-01,469,0.213675,-44.628099


In [27]:
display(core_contribution_share)
print(number_of_regular_users)


,month,core_contribution_share
0,2006-08,50.908430
1,2006-09,59.561776
2,2006-10,67.935232
3,2006-11,63.913137
4,2006-12,60.603039
...,...,...
224,2025-05,51.308655
225,2025-06,52.516221
226,2025-07,55.082761
227,2025-08,53.550116


       month  number_of_regular_users
0    2006-08                       68
1    2006-09                      188
2    2006-10                      258
3    2006-11                      349
4    2006-12                      324
..       ...                      ...
217  2025-05                      375
218  2025-06                      382
219  2025-07                      371
220  2025-08                      369
221  2025-09                      221

[222 rows x 2 columns]
